# 01 — Data Pipeline

**What this notebook does:** turns raw open data from four sources into the model-ready feature table that the training and scoring notebooks consume.

**The four data sources we use** (all open, all French):

| # | Source | Provider | What we get from it |
|---|---|---|---|
| 1 | **INSEE Sirene** | data.gouv.fr (`StockUniteLegale*`) | Company identity over time: name, NAF code, legal form, employees, status (active/cessée), creation/closure dates |
| 2 | **BODACC** | DILA (échanges.dila.gouv.fr) | Legal announcements: procédures collectives, dissolutions, radiations — the strongest distress signals |
| 3 | **INPI RNE bulk** | INPI FTP | Formality filings (radiation, cessation, statutes changes) + annual account filing dates |
| 4 | **Bilans** | data.gouv.fr financial parquet | Revenue, net result, balance-sheet ratios from filed annual accounts |

**The pipeline stages:**

```
[1] DOWNLOAD          source-archives/<source>/...     ← raw files from each provider
[2] RAW EXPORT        data-lake/raw/<source>/*.parquet ← per-source parquet, minimal cleaning
[3] CLEAN BUILD       data-lake/clean/<dataset>/       ← unified column names, typed, deduplicated
[4] FEATURE BUILD     data-lake/features/              ← model-ready company-year rows
[5] SANITY CHECKS                                       ← freshness + counts + per-SIREN diagnostic
```

**You can run a subset** by flipping the `DO_*` flags in the Configuration cell — e.g. to just refresh BODACC without re-downloading everything else.

**You do NOT need to rerun this notebook to retrain the model or score new SIRENs.** Those happen in `02_training.ipynb` and `03_model_interrogation.ipynb`.

## Setup

Mounts Google Drive, pulls the latest repo code, installs the Colab-specific requirements, defines paths. Safe to re-run.

In [ ]:
from pathlib import Path
import os, subprocess, sys

# --- Drive mount (Colab only) ---
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# --- Repo ---
REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'ml-workflow'  # change this if you work on another branch
REPO_DIR = Path('/content/pfein')
BACKEND_DIR = REPO_DIR / 'back_end'

cwd = Path.cwd()
if (cwd / 'collabs' / 'requirements-colab.txt').exists():
    # Already inside the backend (local run) — don't re-clone.
    BACKEND_DIR = cwd
    REPO_DIR = BACKEND_DIR.parent

if not (BACKEND_DIR / 'collabs' / 'requirements-colab.txt').exists():
    if not (REPO_DIR / '.git').exists():
        subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'switch', BRANCH])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])

# --- Paths ---
DRIVE_ROOT = Path('/content/drive/MyDrive/PFE ML Data/pfe_data')
SOURCE_ARCHIVES = DRIVE_ROOT / 'source-archives'  # raw downloaded files (TAZ, parquet, zip, etc.)
DATA_LAKE = DRIVE_ROOT / 'data-lake'              # the processed lake (raw/, clean/, features/)
WORK_DIR = Path('/content/pfe_work')              # local fast SSD for staging, synced back to Drive
DUCKDB_TMP = Path('/content/pfein_duckdb_tmp')    # DuckDB spill-to-disk location

for p in (DRIVE_ROOT, SOURCE_ARCHIVES, DATA_LAKE, WORK_DIR, DUCKDB_TMP):
    p.mkdir(parents=True, exist_ok=True)
os.environ['DUCKDB_TEMP_DIRECTORY'] = str(DUCKDB_TMP)

# --- Python path ---
os.chdir(BACKEND_DIR)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

# --- Requirements ---
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BACKEND_DIR / 'collabs' / 'requirements-colab.txt')])

print(f'BACKEND_DIR    = {BACKEND_DIR}')
print(f'DRIVE_ROOT     = {DRIVE_ROOT}')
print(f'SOURCE_ARCHIVES= {SOURCE_ARCHIVES}')
print(f'DATA_LAKE      = {DATA_LAKE}')
print(f'BRANCH         = {BRANCH}')

## Configuration — what to run

Flip these flags depending on what you want to refresh. **For a first-time setup, leave them all `True`**. For routine refreshes, you typically only need `DO_DOWNLOAD_BODACC = True` (new BODACC announcements come out daily) plus `DO_BUILD_CLEAN` and `DO_BUILD_FEATURES` to propagate them downstream.

In [ ]:
# --- Local SSD staging (recommended) ---
# Google Drive I/O is 10-100x slower than Colab's local SSD for many small parquet
# files. With USE_LOCAL_STAGING=True we mirror data-lake from Drive -> /content/pfe_work
# at session start, run all builds locally, then sync changed files back to Drive.
# Set to False only if you specifically want to read/write Drive directly (slow).
USE_LOCAL_STAGING = True

# --- What to run ---
DO_DOWNLOAD_INSEE   = True
DO_DOWNLOAD_BODACC  = True
DO_DOWNLOAD_INPI    = True   # NEEDS CREDENTIALS — see INPI section
DO_DOWNLOAD_BILANS  = True

DO_BUILD_CLEAN      = True
DO_BUILD_FEATURES   = True

DO_SANITY_CHECKS    = True

# Feature build year range. The default end_year is current_year - 1 so that
# the 12-month label window is fully observed. For scoring-only use cases
# (no training), you can extend end_year to current_year - 1 even mid-year.
FEATURE_START_YEAR  = 2017
FEATURE_END_YEAR    = 2025  # bump this once data through Dec of that year is in the lake
FEATURE_YEAR_BATCH_SIZE = 1   # how many years to build per batch (lower = less RAM, None = all at once)

# --- Mid-year cutoff (optional) ---
# When set, the feature builder ignores FEATURE_START_YEAR/FEATURE_END_YEAR and
# builds a SINGLE row per company with prediction_date = this date. Rolling-window
# features (legal_events_count_12m, etc.) end at this date — fully observed even
# mid-year, no partial-year bias.
#
# Use this to score a known-recent event (e.g. Q1 2026 liquidation) against features
# that reflect 'today'. Output goes to prediction_year=<year of cutoff>; existing
# year-end partitions are NOT touched.
#
# Examples:
#   MID_YEAR_CUTOFF_DATE = None         # default: use year-end behavior above
#   MID_YEAR_CUTOFF_DATE = 'today'      # rebuild a single 'as of today' row per company
#   MID_YEAR_CUTOFF_DATE = '2026-05-17' # specific date
#
# Labels are skipped when this is set (12-month forward window is not yet observable).
# Do NOT train on the partition produced by a mid-year cutoff — its labels are missing.
MID_YEAR_CUTOFF_DATE = None

## Stage data lake to local SSD

If `USE_LOCAL_STAGING = True` (the default), this cell mirrors `data-lake/` from Drive to `/content/pfe_work/data-lake/`. All subsequent build steps read and write the local copy. A final sync cell at the bottom of the notebook pushes the new `clean/` and `features/` partitions back to Drive.

**Why this matters.** Reading thousands of small parquet files from Drive in Colab is brutally slow — a single `read_parquet(glob)` can hang for tens of minutes. The same query against local SSD returns in seconds. The downloads already use the local SSD as a staging area, so this just makes sure builds use it too.

**First-run cost.** The initial rsync transfers the entire data-lake (raw + clean + features). Expect ~5-30 min depending on total size. Subsequent runs are seconds because rsync only transfers new files.

**If `USE_LOCAL_STAGING = False`**: `EFFECTIVE_DATA_LAKE` falls back to Drive. Builds will work but be very slow.

In [ ]:
import shlex

if USE_LOCAL_STAGING:
    EFFECTIVE_DATA_LAKE = WORK_DIR / 'data-lake'
    EFFECTIVE_DATA_LAKE.mkdir(parents=True, exist_ok=True)
    drive_src = str(DATA_LAKE).rstrip('/') + '/'
    local_dst = str(EFFECTIVE_DATA_LAKE).rstrip('/') + '/'
    print(f'Mirroring data-lake: Drive -> local SSD')
    print(f'  source: {drive_src}')
    print(f'  target: {local_dst}')
    print('  (incremental rsync — first run slow, subsequent runs near-instant)')
    cmd = 'rsync -a --info=progress2 ' + shlex.quote(drive_src) + ' ' + shlex.quote(local_dst)
    print(f'$ {cmd}')
    !{cmd}
    print(f'\nEFFECTIVE_DATA_LAKE = {EFFECTIVE_DATA_LAKE}')
else:
    EFFECTIVE_DATA_LAKE = DATA_LAKE
    print(f'Local staging disabled. Operating directly on Drive: {EFFECTIVE_DATA_LAKE}')
    print('Warning: queries against many small parquet files on Drive are very slow.')

## Source 1 — INSEE Sirene

**What it is.** INSEE Sirene is France's national company registry. We download two parquet files from data.gouv.fr:

- `StockUniteLegale_utf8.parquet` — current snapshot of every French SIREN (active or not).
- `StockUniteLegaleHistorique_utf8.parquet` — *period rows* per SIREN, so we can know what the company name / NAF code / legal form / employee bracket / administrative status was on any given date.

**Why we use both.** The current snapshot tells us *what's true right now*. The history file lets us reconstruct *what was true on Dec 31, 2020* — which is what the model needs to build a feature row for `prediction_year=2020`.

**Update frequency.** data.gouv.fr refreshes monthly. Re-download whenever you want the latest active statuses.

**Lands in:** `source-archives/insee/bulk/` (downloads), then `data-lake/raw/insee/bulk/*.parquet` (raw).

**Script called:** [collabs/download_insee.py](../download_insee.py). It only downloads + copies into raw — no parsing needed since INSEE already publishes clean parquet.

In [ ]:
# Real-time output via Colab shell magic (`subprocess.check_call` buffers stdout
# in Colab — you can sit for minutes seeing nothing before output flushes).
import shlex
if DO_DOWNLOAD_INSEE:
    args = [
        'python', '-u', 'collabs/download_insee.py',
        '--drive-root', str(DRIVE_ROOT),
        '--work-dir', str(WORK_DIR),
        '--repo-dir', str(BACKEND_DIR),

    ]
    cmd = ' '.join(shlex.quote(str(a)) for a in args)
    print(f'$ {cmd}')
    !{cmd}
else:
    print('INSEE download skipped (DO_DOWNLOAD_INSEE=False)')

## Source 2 — BODACC

**What it is.** Bulletin Officiel Des Annonces Civiles et Commerciales — the French legal gazette where courts publish company-life events: procédures collectives (sauvegarde, redressement, liquidation), dissolutions, radiations, sales of fonds de commerce, etc.

**Why it matters for our model.** BODACC is the *authoritative* source of distress signals. A company entering liquidation gets a BODACC publication days after the court ruling — no other dataset has this level of timeliness.

**The two modes — and the trap.** DILA publishes BODACC archives in two locations:

- `FluxHistorique/` — past years, in yearly folders. Stable, gets the full picture of e.g. 2024.
- `FluxAnneeCourante/` — the current year, refreshed daily.

**Trap:** `--mode historical` only sees `FluxHistorique/`. If you only run historical mode, you'll get nothing for the current year because DILA hasn't moved it there yet. For a current-as-of-today refresh, you **must** run `--mode current`. This is why earlier feature builds were missing Q1 2026 events.

**Families.**
- `PCL` = Procédures Collectives (sauvegarde, redressement, liquidation, plans de cession) — what we care about most.
- `RCS-B` = dissolutions and radiations.

**Lands in:** `source-archives/bodacc/{current,historical}/<year>/<family>/*.taz`, then parsed to `data-lake/raw/bodacc/*.parquet`.

**Script called:** [collabs/download_bodacc.py](../download_bodacc.py). It does discovery → resumable download → parquet conversion via [app/tools/bodacc_archives_to_parquet.py](../../app/tools/bodacc_archives_to_parquet.py).

In [ ]:
# CURRENT YEAR (FluxAnneeCourante) — run this every time you want fresh BODACC data.
# Real-time output via Colab shell magic (`subprocess.check_call` buffers stdout
# in Colab — you can sit for minutes seeing nothing before output flushes).
import shlex
if DO_DOWNLOAD_BODACC:
    args = [
        'python', '-u', 'collabs/download_bodacc.py',
        '--drive-root', str(DRIVE_ROOT),
        '--work-dir', str(WORK_DIR),
        '--repo-dir', str(BACKEND_DIR),
        '--mode', 'current',
        '--families', 'PCL,RCS-B',
    ]
    cmd = ' '.join(shlex.quote(str(a)) for a in args)
    print(f'$ {cmd}')
    !{cmd}
else:
    print('BODACC current-year download skipped')

In [ ]:
# HISTORICAL BACKFILL (FluxHistorique) — only run this on first-time setup or to
# extend the historical range. Slow (downloads years of archives). Set start/end
# years for the range you want to backfill. Safe to re-run with --skip-existing.
import shlex
DO_BODACC_HISTORICAL_BACKFILL = False  # ← flip to True only on first setup
BACKFILL_START_YEAR = 2017
BACKFILL_END_YEAR   = 2025

if DO_BODACC_HISTORICAL_BACKFILL:
    args = [
        'python', '-u', 'collabs/download_bodacc.py',
        '--drive-root', str(DRIVE_ROOT),
        '--work-dir', str(WORK_DIR),
        '--repo-dir', str(BACKEND_DIR),
        '--mode', 'historical',
        '--families', 'PCL,RCS-B',
        '--start-year', str(BACKFILL_START_YEAR),
        '--end-year', str(BACKFILL_END_YEAR),
    ]
    cmd = ' '.join(shlex.quote(str(a)) for a in args)
    print(f'$ {cmd}')
    !{cmd}
else:
    print('BODACC historical backfill skipped (DO_BODACC_HISTORICAL_BACKFILL=False)')

## Source 3 — INPI RNE bulk

**What it is.** The INPI Registre National des Entreprises publishes a bulk FTP feed of all formality filings (creation, modification of statutes, radiation, cessation, etc.) and all annual account filings.

**Why we use it.**
- *Formalities* tell us about company-life events that haven't yet hit BODACC (INPI is sometimes faster than BODACC).
- *Annual accounts* tell us *when* a company last filed — companies that stop filing are often in distress.

**Caveat — current schema is lossy.** The clean pipeline currently extracts only generic `event_type` and `event_text` for formalities. The actual *type* of a formality (radiation_definitive, cessation_activite, etc.) lives deeper in the raw nested JSON and isn't extracted. So today we know *that* a company filed something in 2026, but not *what*. This is a known limitation — see the diagnostic at the end.

**Credentials.** INPI requires FTP credentials. Set them as env vars before running this cell, or pass via `--user` / `--password` flags:

```python
os.environ['INPI_FTP_HOST']     = '...'  # provided by INPI
os.environ['INPI_FTP_USER']     = '...'
os.environ['INPI_FTP_PASSWORD'] = '...'
```

**Lands in:** `source-archives/inpi/{formalites,comptes_annuels}/` (downloads), then `data-lake/raw/inpi/{formalites,comptes_annuels}/*.parquet`.

**Script called:** [collabs/download_inpi.py](../download_inpi.py).

In [ ]:
# Real-time output via Colab shell magic (`subprocess.check_call` buffers stdout
# in Colab — you can sit for minutes seeing nothing before output flushes).
import shlex
if DO_DOWNLOAD_INPI:
    # Sanity check: bail early with a clear message if credentials missing.
    if not (os.environ.get('INPI_FTP_HOST') and os.environ.get('INPI_FTP_USER') and os.environ.get('INPI_FTP_PASSWORD')):
        raise RuntimeError(
            'INPI credentials missing. Set INPI_FTP_HOST / INPI_FTP_USER / INPI_FTP_PASSWORD env vars, '
            'or set DO_DOWNLOAD_INPI=False to skip this source.'
        )

    args = [
        'python', '-u', 'collabs/download_inpi.py',
        '--drive-root', str(DRIVE_ROOT),
        '--work-dir', str(WORK_DIR),
        '--repo-dir', str(BACKEND_DIR),
        '--categories', 'formalites,comptes_annuels',
        '--niveaux', 'standard,niveau1',
    ]
    cmd = ' '.join(shlex.quote(str(a)) for a in args)
    print(f'$ {cmd}')
    !{cmd}
else:
    print('INPI download skipped (DO_DOWNLOAD_INPI=False)')

## Source 4 — Bilans (consolidated financials)

**What it is.** A single consolidated parquet on data.gouv.fr (`donnees-financieres-detaillees-des-entreprises-format-parquet`) containing balance-sheet and income-statement figures from filed annual accounts, across all French companies, all years.

**Why we use it.** INPI gives us *when* a company filed (filing date); this gives us *what* they filed (revenue, net result, debt ratios). These are strong predictors of distress.

**Update frequency.** Cumulative refresh — re-downloading replaces the file. Pull whenever you want fresher revenue/net result figures.

**Lands in:** `source-archives/financials/data_gouv/` → `data-lake/raw/financials/*.parquet`.

**Script called:** [collabs/download_bilan.py](../download_bilan.py).

In [ ]:
# Real-time output via Colab shell magic (`subprocess.check_call` buffers stdout
# in Colab — you can sit for minutes seeing nothing before output flushes).
import shlex
if DO_DOWNLOAD_BILANS:
    args = [
        'python', '-u', 'collabs/download_bilan.py',
        '--drive-root', str(DRIVE_ROOT),
        '--work-dir', str(WORK_DIR),
        '--repo-dir', str(BACKEND_DIR),

    ]
    cmd = ' '.join(shlex.quote(str(a)) for a in args)
    print(f'$ {cmd}')
    !{cmd}
else:
    print('Bilans download skipped (DO_DOWNLOAD_BILANS=False)')

## Build clean datasets

**Why a clean stage exists.** Each raw source has its own column names, types, and quirks: BODACC has `dateParution`, INPI has `date_depot`, INSEE has `date_debut_periode`, etc. The clean stage normalizes everything into a small unified schema (`siren`, `event_date`, `event_type`, plus dataset-specific flags) so the feature builder can write portable SQL.

**What gets built.** Four parquet datasets under `data-lake/clean/`:

| Clean dataset | Built from | Used by feature builder for |
|---|---|---|
| `company_identity/` | INSEE Stock + Historique | identity columns at the cutoff date |
| `legal_events/` | BODACC parquet | `legal_events_count_*`, `flag_liquidation`, etc. |
| `formalities_events/` | INPI formalities | INPI radiation / cessation detection |
| `annual_accounts/` | INPI annual accounts | `annual_accounts_count_*`, missed-filing detection |

**Financials** are *not* cleaned (the raw data.gouv parquet is already usable as-is).

**Function called:** [`app.tools.build_clean_core_sources.build_clean_datasets`](../../app/tools/build_clean_core_sources.py). Pure Python — you can read the implementation directly.

In [ ]:
if DO_BUILD_CLEAN:
    from app.tools.build_clean_core_sources import build_clean_core_sources
    from app.tools.build_clean_financials import build_clean_financials

    summary = build_clean_core_sources(
        data_lake_dir=EFFECTIVE_DATA_LAKE,
        overwrite=False,    # set True to wipe and rebuild from scratch (slow)
        max_rows=None,      # set e.g. 10_000 for a smoke test
    )
    summary["financials"] = build_clean_financials(
        data_lake_dir=EFFECTIVE_DATA_LAKE,
        overwrite=False,
        max_rows=None,
    )

    print("Clean build summary:")
    for name, info in summary.items():
        rows = info.get("rows", "n/a")
        reason = info.get("skipped_reason")
        if reason:
            print(f"  {name}: SKIPPED ({reason})")
        else:
            print(f"  {name}: rows={rows}")
else:
    print("Clean build skipped (DO_BUILD_CLEAN=False)")

## Build company-year features

**What this produces.** One row per `(siren, prediction_year)` with ~50 numerical/categorical features describing the company as of Dec 31 of `prediction_year`. This is what the model actually consumes.

**Examples of feature columns.**
- Identity: `company_name`, `activity_code`, `legal_category_code`, `administrative_status_at_cutoff`, `company_age_years`, `employee_size_bracket`.
- Legal events: `legal_events_count_12m`, `legal_risk_events_count_12m`, `days_since_last_legal_event`.
- Filings: `annual_accounts_count_24m`, `days_since_last_account_filing`, `years_since_last_financial_statement`.
- Financials: `latest_revenue`, `latest_net_result`, `latest_debt_to_assets`.

**Also produces labels** — `risk_labels/` — capturing whether each company stopped being active in the 12 months *after* its cutoff. These are only used for training; scoring ignores them.

**Output partitions:** `data-lake/features/company_year_features/prediction_year=YYYY/*.parquet`. Re-running for a single year only rewrites that partition (other years are preserved) — *unless* you pass `--overwrite`, which wipes everything.

**Function called:** [`app.tools.build_company_year_features.build_company_year_datasets`](../../app/tools/build_company_year_features.py).

In [ ]:
if DO_BUILD_FEATURES:
    from datetime import date
    from app.tools.build_company_year_features import build_company_year_datasets

    cutoff_date_arg = None
    if MID_YEAR_CUTOFF_DATE:
        if str(MID_YEAR_CUTOFF_DATE).lower() == 'today':
            cutoff_date_arg = date.today()
        else:
            cutoff_date_arg = date.fromisoformat(MID_YEAR_CUTOFF_DATE)
        print(f'Mid-year cutoff mode: building single partition for prediction_date={cutoff_date_arg}')
        print('(FEATURE_START_YEAR/FEATURE_END_YEAR are ignored; labels are skipped.)')
    else:
        print(f'Year-end mode: building partitions for {FEATURE_START_YEAR}..{FEATURE_END_YEAR}')

    build_company_year_datasets(
        data_lake_dir=EFFECTIVE_EFFECTIVE_DATA_LAKE,
        start_year=FEATURE_START_YEAR,
        end_year=FEATURE_END_YEAR,
        max_companies=None,   # set e.g. 100_000 for a smoke test
        year_batch_size=FEATURE_YEAR_BATCH_SIZE,
        overwrite=False,      # False = add/replace only the year partitions in this range
        cutoff_date=cutoff_date_arg,
        features_only=bool(cutoff_date_arg),
    )
    print('
Feature build done.')
else:
    print('Feature build skipped (DO_BUILD_FEATURES=False)')

## Sanity checks

Three things to verify after a pipeline run:

1. **Dataset freshness** — is each source actually current? If BODACC's most recent `event_date` is 6 months old, no model can detect recent distress.
2. **Row counts** — did each clean dataset get populated? An empty `legal_events` table is a silent failure.
3. **Per-SIREN inspection** — for a specific company, did the pipeline pick up the events you expect? Useful for validating downstream model behavior.

In [ ]:
if DO_SANITY_CHECKS:
    import duckdb
    import pandas as pd

    def _glob(*candidates):
        for c in candidates:
            c = Path(c)
            if c.exists() and any(c.rglob('*.parquet')):
                return (c / '**' / '*.parquet').as_posix().replace("'", "''"), c
        return None, None

    DATASETS = {
        'company_identity': (EFFECTIVE_DATA_LAKE / 'clean' / 'company_identity',
                             EFFECTIVE_DATA_LAKE / 'raw' / 'insee' / 'bulk' / 'stock_unite_legale',
                             'period_start'),
        'legal_events':     (EFFECTIVE_DATA_LAKE / 'clean' / 'legal_events',
                             EFFECTIVE_DATA_LAKE / 'raw' / 'bodacc',
                             'event_date'),
        'formalities':      (EFFECTIVE_DATA_LAKE / 'clean' / 'formalities_events',
                             EFFECTIVE_DATA_LAKE / 'raw' / 'inpi' / 'formalites',
                             'event_date'),
        'annual_accounts':  (EFFECTIVE_DATA_LAKE / 'clean' / 'annual_accounts',
                             EFFECTIVE_DATA_LAKE / 'raw' / 'inpi' / 'comptes_annuels',
                             'filing_date'),
        'financials':       (EFFECTIVE_DATA_LAKE / 'raw' / 'financials',
                             None,
                             None),
        'features':         (EFFECTIVE_DATA_LAKE / 'features' / 'company_year_features',
                             None,
                             'prediction_year'),
    }

    rows = []
    con = duckdb.connect()
    try:
        for name, (primary, fallback, freshness_col) in DATASETS.items():
            glob_sql, resolved_dir = _glob(*(c for c in (primary, fallback) if c is not None))
            if not glob_sql:
                rows.append({'dataset': name, 'resolved': 'NOT FOUND', 'row_count': None, 'freshness': None})
                continue
            row_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{glob_sql}', union_by_name=true)").fetchone()[0]
            freshness = None
            if freshness_col:
                try:
                    freshness = con.execute(
                        f"SELECT max({freshness_col}) FROM read_parquet('{glob_sql}', union_by_name=true)"
                    ).fetchone()[0]
                except Exception as exc:
                    freshness = f'ERR: {exc}'
            rows.append({
                'dataset': name,
                'resolved': str(resolved_dir.relative_to(EFFECTIVE_DATA_LAKE)),
                'row_count': row_count,
                f'max_{freshness_col}' if freshness_col else 'freshness': freshness,
            })
    finally:
        con.close()

    df = pd.DataFrame(rows)
    print('Data lake summary:')
    display(df)
else:
    print('Sanity checks skipped (DO_SANITY_CHECKS=False)')

### Per-SIREN diagnostic

Drop a SIREN below to see exactly what the data lake contains for that company. Useful when validating: 'I know company X is in liquidation — does the pipeline see it?'

In [ ]:
import re
import duckdb
import pandas as pd

def diagnose_siren(siren_value):
    siren = re.sub(r'\D', '', str(siren_value))
    if len(siren) != 9:
        raise ValueError(f'SIREN must be 9 digits, got {siren_value!r}')
    siren_lit = "'" + siren + "'"

    def _glob(*candidates):
        for c in candidates:
            c = Path(c)
            if c.exists() and any(c.rglob('*.parquet')):
                return (c / '**' / '*.parquet').as_posix().replace("'", "''")
        return None

    ident_sql = _glob(EFFECTIVE_DATA_LAKE / 'clean' / 'company_identity')
    legal_sql = _glob(EFFECTIVE_DATA_LAKE / 'clean' / 'legal_events')
    form_sql  = _glob(EFFECTIVE_DATA_LAKE / 'clean' / 'formalities_events')
    acc_sql   = _glob(EFFECTIVE_DATA_LAKE / 'clean' / 'annual_accounts')
    feat_sql  = _glob(EFFECTIVE_DATA_LAKE / 'features' / 'company_year_features')

    con = duckdb.connect()
    try:
        print(f'=== SIREN {siren} ===\n')

        if ident_sql:
            print('-- INSEE identity (most recent period first) --')
            df = con.execute(f"""
                SELECT company_name, activity_code, administrative_status, period_start, period_end, closure_date
                FROM read_parquet('{ident_sql}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                ORDER BY COALESCE(period_start, DATE '1900-01-01') DESC LIMIT 5
            """).df()
            display(df)

        if legal_sql:
            print('\n-- BODACC events (most recent first) --')
            df = con.execute(f"""
                SELECT event_date, event_category, event_type, is_radiation,
                       flag_liquidation, flag_redressement, flag_sauvegarde, flag_procedure_collective
                FROM read_parquet('{legal_sql}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                ORDER BY event_date DESC LIMIT 20
            """).df()
            print(f'  ({len(df)} rows)')
            if not df.empty: display(df)

        if form_sql:
            print('\n-- INPI formalities (most recent first) --')
            df = con.execute(f"""
                SELECT event_date, event_type, event_text
                FROM read_parquet('{form_sql}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                ORDER BY event_date DESC LIMIT 10
            """).df()
            print(f'  ({len(df)} rows)')
            if not df.empty: display(df)

        if acc_sql:
            print('\n-- Annual accounts filings --')
            df = con.execute(f"""
                SELECT filing_date FROM read_parquet('{acc_sql}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                ORDER BY filing_date DESC LIMIT 10
            """).df()
            print(f'  ({len(df)} rows)')
            if not df.empty: display(df)

        if feat_sql:
            print('\n-- Pre-built feature rows --')
            df = con.execute(f"""
                SELECT prediction_year, administrative_status_at_cutoff,
                       legal_events_count_12m, legal_risk_events_count_12m,
                       days_since_last_legal_event, annual_accounts_count_24m,
                       latest_revenue, latest_net_result
                FROM read_parquet('{feat_sql}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                ORDER BY prediction_year DESC
            """).df()
            print(f'  ({len(df)} rows)')
            if not df.empty: display(df)
    finally:
        con.close()

# Example:
# diagnose_siren('444560502')

## Sync local SSD → Drive

When `USE_LOCAL_STAGING = True`, the build steps wrote `clean/` and `features/` partitions to the local SSD. This cell pushes them back to Drive so the next session (or a different notebook) can read them.

`raw/` is already on Drive — the download scripts sync it during their own run. We only need to push the *derived* datasets here.

`rsync` is incremental: only files that changed since the last sync are transferred. Re-running this cell is cheap.

In [ ]:
import shlex

if USE_LOCAL_STAGING and EFFECTIVE_DATA_LAKE != DATA_LAKE:
    for subpath in ('clean', 'features'):
        local_src = EFFECTIVE_DATA_LAKE / subpath
        drive_dst = DATA_LAKE / subpath
        if not local_src.exists():
            print(f'  skipping {subpath}/ — nothing local to sync')
            continue
        drive_dst.mkdir(parents=True, exist_ok=True)
        src_str = str(local_src).rstrip('/') + '/'
        dst_str = str(drive_dst).rstrip('/') + '/'
        print(f'Syncing local {subpath}/ -> Drive')
        cmd = 'rsync -a --info=progress2 ' + shlex.quote(src_str) + ' ' + shlex.quote(dst_str)
        print(f'$ {cmd}')
        !{cmd}
        print()
    print('Sync to Drive complete.')
else:
    print('Local staging disabled or already pointing at Drive — no sync needed.')

## Done

If the sanity checks above show non-empty datasets with recent `max(event_date)` / `max(filing_date)` / `max(prediction_year)` matching what you expect, the lake is healthy.

**Next steps:**

- `02_training.ipynb` — train (or retrain) the continuity-risk model from the features built here.
- `03_model_interrogation.ipynb` — load a trained model and score SIRENs interactively.

**Common refresh workflow** (once the lake is initially populated):

1. Set `DO_DOWNLOAD_INSEE = DO_DOWNLOAD_INPI = DO_DOWNLOAD_BILANS = False`, keep `DO_DOWNLOAD_BODACC = True` (most frequently updated).
2. Keep `DO_BUILD_CLEAN = DO_BUILD_FEATURES = True` so the new BODACC events flow through.
3. Bump `FEATURE_END_YEAR` if you've crossed into a new calendar year and want to score against the latest cutoff.